<a href="https://colab.research.google.com/github/data-188-berkeley/hw4/blob/main/hw4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data 188 HW4

In this assignment, you will train a transformer encoder+decoder model to perform **machine translation**.
Your model will learn from example, as we provide you with (text_src, text_tgt) pairs.
We will train on the [Multi30K](https://arxiv.org/abs/1605.00459) dataset, which consists of 31K (text_english, text_german) pairs.

Note that, while Multi30K contains both text and image signals, we'll only use the text signals for this assignment to keep things simple.

You will do this using a Transformer network, from the __[Attention is all you need](https://arxiv.org/abs/1706.03762)__ paper.
In this assignment you will:
- Process raw text into token ids via tokenizers.
- Implement the key conceptual blocks of a Transformer.
- Train a Transformer model to perform machine translation.

** Before you start **

You should familiarize yourself with the Transformer model architecture.
If you'd like, you can also read the original Transformer paper (linked above).

We are providing you with skeleton code for the Transformer, but there will have to implement 5 conceptual blocks of the transformer yourself:
- `AttentionQKV`: the Query, Key, Value attention mechanism at the center of the Transformer
- `MultiHeadAttention`: the multiple heads that enable each input to attend at many places at once.
- `PositionEmbedding`: the sinusoid-based position embedding of the Transformer.
- Encoder & Decoder: The encoder (that reads the source sequence, eg English text), the decoder (that produces the target sequence, eg German text)
- Full Transformer: piecing it all together.

## Clarifications Doc

In case we need to give any clarifications (or hints!) for the assignment, see this Google Doc: ["HW4 Clarifications"](https://docs.google.com/document/d/16WtRwEljgxdwz0XTDaKp4luhNjJ5xD_CY30qPusmVbY/edit?tab=t.0).
This doc will be regularly updated.

## General Tips

Unless otherwise instructed, don't modify cells aside from in between the blocks with `# BEGIN YOUR SOLUTION`, `# END YOUR SOLUTION`.

Modifying other code can result in failing autograder tests.

**CPU vs GPU**: We recommend you to start work on this notebook using a CPU Colab instance at first.
Then, switch over to a GPU instance (T4) only when the instructions tell you to do so (namely, when it's time to train a larger-scale model).
This is to avoid running out of GPU compute.

In [ ]:
# Run this cell each time your kernel is disconnected or restarted
# Tip: it's always safe to re-run this, this will never delete data

import os
import sys

# basedir_course: All colab material for this course will live here.
#   Feel free to modify this if you'd like.
basedir_course = "/content/drive/MyDrive/data188"
# asn_name: name of assignment, eg hw4. This must match github repo name.
asn_name = "hw4"
rootdir_asn = os.path.join(basedir_course, asn_name)

# Fetch code to set up the assignment, then set the correct working directory
# (eg for python imports to work)
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(basedir_course, exist_ok=True)
os.chdir(basedir_course)
os.system(f"git clone https://github.com/data-188-berkeley/{asn_name}.git")
os.chdir(rootdir_asn)

# install required libraries
!pip3 install torchmetrics

# Validate that our current working directory is correct.
# This should output:
#   /content/drive/MyDrive/data188/hw4/
print("Current working directory: ", os.getcwd())
# Another check: let's double check that the files in hw4/ are in the current directory.
# This should output something like:
#   ['.git', 'LICENSE', 'apps', 'data', 'hw4.ipynb', 'tests', '.pytest_cache']
print("ls cwd: ", os.listdir(os.getcwd()))

# ==== Important: modify path to allow our imports to work to work ====
# Implementation note: Annoyingly, we can't append to an existing env-var via
# ipython `set_env` like we can in bash (eg PYTHONPATH=new_path/:$PYTHONPATH).
# Instead we assume that PYTHONPATH starts as: /env/python, eg the below
# line is hopefully equivalent to:
#   PYTHONPATH=./python/:$PYTHONPATH
%set_env PYTHONPATH ./:/env/python
# We also need to modify sys.path directly so that the current python session
# can access our code
sys.path.insert(0, "./")

# outdir_base: where all generated data lives, including: datasets, tokenizers, and model snapshots
outdir_base = os.path.join(rootdir_asn, "./data/")

print("Setup done!")

# Tokenizers Demo
Recall that, when working with text in AI/ML, we frequently convert raw text human-readable strings into numeric representation via tokenization.
In this assignment, we'll use the [SentencePiece](https://huggingface.co/docs/transformers/en/tokenizer_summary#sentencepiece) tokenizer, a popular tokenizer that supports multiple languages (including English and German).

We'll load the tokenizer from Huggingface: notably, we'll use the same tokenizer that the [T5](https://huggingface.co/docs/transformers/en/model_doc/t5) model used.

In [ ]:
from utils_public.utils_tokenizer import get_tokenizer_t5

tokenizer, tokenizer_meta = get_tokenizer_t5(
    cache_dir=os.path.join(outdir_base, "tokenizers", "hf_tokenizers"),
    strip_case=True,
)
print(f"tokenizer_meta: {tokenizer_meta}")

# Demo: tokenize some text
input_texts = [
    "This is an example sentence",
    "Using tokenizers in pytorch and huggingface is easy!",
    "Dieser Tokenizer unterstützt mehrere Sprachen, wie zum Beispiel Deutsch.",
    "Ce tokenizer prend également en charge le français.",
    "¡Finalmente, esta tokenizer también soporta español!",
]

# we'll tokenize the above sentences, enforcing that max_length=16, and
# padding all sequences to len=16 (with possible truncation)
# shape=[b, max_seq_len]
encoded_texts = tokenizer(
    input_texts,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=32,
)

# Important keys:
# "input_ids": shape=[b, seq_len]. Token ids
# "attention_mask": shape=[b, seq_len]. Padding mask. `1` means "valid" (not pad), `0` means pad.
print(f"encoded_texts: {encoded_texts}")

In [ ]:
# back to human-readable texts. note that this contains the special tokens
# like: [EOS] ("</s>"), [PAD] ("<pad>")
decoded_texts = tokenizer.decode(
    encoded_texts["input_ids"],
)

print(f"decoded_texts: {decoded_texts}")

In [ ]:
# to omit the special tokens, we can pass skip_special_tokens=True:
decoded_texts_2 = tokenizer.decode(
    encoded_texts["input_ids"],
    skip_special_tokens=True,
)
print(f"decoded_texts_2: {decoded_texts_2}")

In [ ]:
# Note that "attention_masks" is the padding mask that tell us which tokens are valid:
#   `0`: token is invalid (ex: a [PAD] token)
#   `1`: token is valid
# we can validate this by masking out the invalid tokens and decoding:
token_ids_0 = encoded_texts["input_ids"][0, :]
attention_mask_0 = encoded_texts["attention_mask"][0, :]

print(f"token_ids_0: {token_ids_0}")
print(f"attention_mask_0: {attention_mask_0}")
print(f"Without masking via attention_mask: {tokenizer.decode(token_ids_0)}")
token_ids_0_masked = token_ids_0[attention_mask_0 == 1]
print(f"With masking via attention_mask: {tokenizer.decode(token_ids_0_masked)}")

In [ ]:
# Let's view all "special tokens" that this tokenizer supports:
# Fun fact: the "UNKNOWN" token is a special token to accommodate input text
# that the tokenizer can't recognize.
print(tokenizer.special_tokens_map)
for special_token_name, token_str in tokenizer.special_tokens_map.items():
    token_id_int = tokenizer.vocab[token_str]
    print(f"({special_token_name}) {token_str} (token_id={token_id_int})")

## Regarding BOS, EOS, PAD

In many transformer encoder+decoder explanations (including in our class), we frame machine translation as a task like:
```python
X_src = [<BOS>, "I", "am", "Eric", <EOS>]
X_tgt = [<BOS> "Je", "suis", "Eric"]  # right-shifted decoder input
Y_tgt = ["Je", "suis", "Eric", <EOS>]
```

However, in this homework assignment, our tokenizer does not have a `BOS` special token, only `PAD` and `EOS`!

The way this implementation handles this is: our right-shifted decoder input will prepend a `PAD` token (rather than a `BOS` token):
```python
X_src = ["I", "am", "Eric", <EOS>]
X_tgt = [<PAD> "Je", "suis", "Eric"]  # right-shifted decoder input, with PAD
Y_tgt = ["Je", "suis", "Eric", <EOS>]
```

# BLEU scores, SacreBLEU

[BLEU](https://en.wikipedia.org/wiki/BLEU) scores are a popular metric used in NLP to compare the quality of predicted generated text against ground-truth text.
The score ranges from `[0.0, 1.0]`, where `0.0` is lowest quality, and `1.0` is highest quality (exact match).

[SacreBLEU](https://github.com/mjpost/sacrebleu) is a popular library aimed at standardizing the methodology of calculating BLEU scores.
Fortunately, `torchmetrics` contains a nice implementation of SacreBLEU, called [torchmetrics.text.SacreBLEUScore](https://lightning.ai/docs/torchmetrics/stable/text/sacre_bleu_score.html), which we'll use in this assignment.

To get a better intuition of SacreBLEU scores, let's see how it behaves with different pairs of pred/gt text:

In [ ]:
from torchmetrics.text import SacreBLEUScore

# Create SacreBLEUScore instance, set to ignore case
sacre_bleu = SacreBLEUScore(lowercase=True)

# pairs of predicted/ground-truth text
sentence_pairs = [
    (
        # Predicted sentence
        "The singer is singing a song to the audience.",
        # Ground-truth sentence
        "The singer is singing a song to the audience.",
    ),
    (
        # Predicted sentence
        "The artist is singing a song to the audience.",
        # Ground-truth sentence
        "The singer is singing a song to the audience.",
    ),
    (
        # Predicted sentence
        "I'm going for a nice walk to the downtown.",
        # Ground-truth sentence
        "The singer is singing a song to the audience.",
    ),
    (
        # Predicted sentence
        "The vocalist is serenading the audience with a nice song.",
        # Ground-truth sentence
        "The singer is singing a song to the audience.",
    ),
]

for pred_sent, gt_sent in sentence_pairs:
    print(f"(pred) {pred_sent} (gt) {gt_sent} sacreBLEU(pred, gt): {sacre_bleu([pred_sent], [[gt_sent]])}")


Play around with the above cell to get a sense for how SacreBLEU behaves, ie by adding more examples.

You'll find that, while SacreBLEU is a decent proxy for measuring alignment between two sentences, it's imperfect.
However, in practice it's a useful metric because it's easy to calculate, doesn't require human labelers, and is a much better metric than CrossEntropyLoss for tasks like machine translation.

# Transformer implementation

You will implement each part of the Transformer. To finish this section, you should get a very small error (ex: <= 1e-4) on each of the checks in this section.

The Transformer implementation is split into 3 files: `transformer_attention.py`, `transformer_utils.py`, and `transformer.py`.

Each section below gives you directions and a way to verify your code works properly.

You do not need to modify the rest of the code provided, but should read it to understand overall architecture.

## (Question 1) Query-Key-Value Attention (AttentionQKV)

This part is located in `AttentionQKV` in `transformer_attention.py`. You must implement the `AttentionQKV::forward()` function of the class.
You will need to implement the mathematical procedure of AttentionQKV that is described in the [Attention is all you need paper](https://arxiv.org/pdf/1706.03762.pdf).

The following cell will test your implementation.

In [ ]:
!python -m pytest -v -s -k "attention_qkv"

## (Question 2) Multi-head attention

This part is located in the class `MultiHeadProjection` in `transformer_attention.py`.
You must implement the following functions:
- `MultiHeadProjection::_split_heads()`
- `MultiHeadProjection::_combine_heads()`

**Procedure**

The objective is to leverage the `AttentionQKV` class you already wrote.

Your input are the queries, keys, values as 3-d tensors (batch_size, sequence_length, feature_size).

Split them into 4-d tensors (batch_size, n_heads, sequence_length, new_feature_size). Where:
$$feature\_size = n\_heads * new_{feature\_size}.$$

You can then feed the split qkv to your implemented AttentionQKV, which will treat each head as an independent attention function.

Then the output must be combined back into a 3-d tensor.
You can test the validity of your implementation in the cell below.

In [ ]:
!python -m pytest -v -s -k "multihead_projection"

## (Question 3) Position Embedding, Feed Forward (FFN)

Next, implement the `TransformerFeedForward` and `PositionEmbedding` classes in `transformer.py`.

The cell below helps you verify the validity of your implementation

In [ ]:
!python -m pytest -v -s -k "position_embedding"

## (Question 4) Transformer Encoder, Decoder

You now have all the blocks needed to implement the Transformer encoder/decoder blocks.
For this part, fill in 2 classes in the `transformer.py` file: `TransformerEncoderBlock`, `TransformerDecoderBlock`.

The code below will verify the accuracy of each block

In [ ]:
!python -m pytest -v -s -k "transformer_encoder_block"

In [ ]:
!python -m pytest -v -s -k "transformer_decoder_block"

## (Question 5) Transformer

This is the final high-level function that pieces it all together.

Complete the `Transformer::forward()` function in the `transformer.py` file.

The block below verifies your implementation is correct.

In [ ]:
!python -m pytest -v -s -k "transformer_model"

# Training the model

Next, you will train your Transformer encoder+decoder model on the Multi30k EN->DE training dataset.

## (Sanity check) Overfit to a single example

To double check that our pipeline is implemented correctly (dataloader, model architecture, train/test loop), we will first check if we can overfit to a single training dataset example.

It turns out that the first training dataset example is (note: truncated due to `max_seq_len=16` for this section)):
```
Input: two young, white males are outside near many bushes.
GT: zwei junge weiße männer sind im freien in der
```

The `--dbg_try_overfit` will modify our training dataloader to repeatedly loop over this first training dataset example (already implemented for you).

Run the below cell to train a small transformer encoder+decoder model on this dataset.
You should see training loss approach `0.0`, and the resulting outputs should show that:
- Validation metrics are very bad
- Test metrics are very bad

In fact, in the model prediction visualizations, you should see that the model always outputs "zwei junge weiße männer sind im freien in der": in other words, it has completely overfit to the training set!

Believe it or not, this is actually a good sign: it's a sign that our pipeline is implemented correctly.
If, on the other hand, our model couldn't overfit to even a single training example, it's likely that we have a bug somewhere in our pipeline.

If you're not seeing the above, then revisit your implementation to see if you've made a mistake.

Finally: take time to understand the training stdout outputs, and read through the `main_train_multi30k.py` to make sure you understand what the code is doing.
Be sure to take a peek at the train/val/test `.png` plots that `main_train_multi30k.py` generates for you in `data/model_snapshots/`.

In [ ]:
# Kick off training run on a small model size, try to overfit to a single training
# data example.
# Important: leave all below hyperparameters unchanged!
# This should finish in ~180 secs on a CPU instance, and should get:
#   train_error: 0.0049
#   val_error: 57.2
#   test_error: 57.4
# It turns out that the error metric (ie CrossEntropyLoss) is a poor proxy for translation
# quality. Thus, we also use the SacreBLEU metric as a better signal for translation quality,
# your model should achieve the following SacreBLEU scores:
#   val_sacre_bleu_score: 0.0000
#   test_sacre_bleu_score: 0.0030
!python main_train_multi30k.py \
--model_size small_v2 \
--num_train_epochs 100 \
--batchsize_train 32 \
--max_seq_len 16 \
--test_max_output_len 16 \
--use_amp \
--visualize_every_n_steps 100 \
--val_every_n_epochs -1 \
--skip_save_model \
--device cpu \
--dbg_try_overfit

## Train a model (GPU)

Next, we'll train a medium-sized transformer model on a moderate training schedule (30) epochs.
You should be able to train a somewhat reasonable translation model!

For this, switch to a GPU instance type (ex: T4) before running this cell.

**Important**: when you do switch to a new GPU instance, you will lose all local variables.
Be sure to run the first cell (top of this notebook) to ensure that things are setup correctly (ex: working dir is correct, and important vars like `asn_name` are correctly set).
**If you don't do this, the assignment will break!**

Note: to keep train times low for this assignment, we intentionally set the model architecture and train schedule very low.
For reference, training large transformer models on large datasets can take days (or weeks!) of training time.

Your model snapshots (as well as figures like loss curves) will be saved to: `hw4/data/model_snapshots/`, as `.pt` files.
Feel free to take a peek in this directory after your training run is complete (in particular, view the loss curves in the generate .png file!).

In [ ]:
# Next, switch to a GPU instance type, and run this block
# Important: leave ALL below hyperparameters unchanged!
# Kick off training run on medium model size, with longer train schedule (30 epochs)
# This should finish within ~420 secs on a Colab T4 GPU instance, and should get something like:
#   train_loss: 1.8983
#   val_loss: 2.282
#   test_loss: 2.2769
# It turns out that the loss metric (ie CrossEntropyLoss) is a poor proxy for translation
# quality. Thus, we also use the SacreBLEU metric as a better signal for translation quality,
# your model should achieve something like the following SacreBLEU scores:
#   val_sacre_bleu_score: 0.2958
#   test_sacre_bleu_score: 0.2840
# Note that some variation in train/val/test loss/metrics is expected.
!python main_train_multi30k.py \
--model_size large_v2 \
--num_train_epochs 30 \
--batchsize_train 128 \
--max_seq_len 32 \
--test_max_output_len 32 \
--use_amp \
--visualize_every_n_steps 100 \
--val_every_n_epochs 10 \
--device cuda

## (For fun) (Interactive) Talk to your model!

For fun, run the below cell to launch an interactive session where you ask your model to translate text of your choice!
This cell will load the model snapshot from the previous cell (the final epoch snapshot).

Try it out for a few examples.

In [ ]:
%run main_interactive_translate.py \
--path_to_model_weights "./data/model_snapshots/large_v2_multi30k_ba8c6647e7a04728_weights.pth" \
--path_to_model_meta "./data/model_snapshots/large_v2_multi30k_ba8c6647e7a04728_meta.pth" \
--device cpu

In [ ]:
# or, if you prefer, you can call it from the CLI like so
!python main_interactive_translate.py \
--path_to_model_weights "./data/model_snapshots/large_v2_multi30k_ba8c6647e7a04728_weights.pth" \
--path_to_model_meta "./data/model_snapshots/large_v2_multi30k_ba8c6647e7a04728_meta.pth" \
--device cpu \
--query 'a man in a vest is sitting in a chair and holding a cake.'

You will likely find that the model's translations are somewhat disappointing!
In fact, many translations will likely output nonsense.
But, some of them will look reasonable.

Yet, if you were to copy+paste English queries from Multi30k, you'd likely see that the translations are noticeably better than us typing in free-form English text queries.

Why is there such a mismatch between Multi30k performance, and our personal interactions with the model?

Hint: revisit the [Multi30k dataset](https://arxiv.org/abs/1605.00459), and look at some example dataset pairs (EN, DE).
Think about the nature (data distribution) of the dataset, and whether the data distribution would match, say, free-form queries from a user (Hint: it has to do with domain shift).

Given this epiphany, try issuing some new "Multi30k-like" queries to the translation model to see if you can generate higher-quality translations.

Note that our poor translation quality can also be explained by the fact that (1) we're training on a very small dataset, and (2) we're training a fairly small transformer model with a short train schedule.
That being said, the above domain-shift point still stands.

Note: no need to submit a written answer for this question, but you should ponder this!

# (Epilogue) Code exploration, final thoughts

Congratulations on finishing the assignment!
While you implemented many of the important modeling components of the transformer encoder+decoder, there are still several additional important details required to build a machine translation pipeline.

Spend some time reading through the code to answer these questions:

(1) During inference (ie during the interactive session), how does autoregressive inference work? Hint: check out `utils_public/utils_translation_inference.py`

(2) Autoregressive inference is only done at test time (ie to calculate val/test metricS).
Why don't we need to do autoregressive inference during training?

(3) Why is classification accuracy a poor proxy for translation quality? And why do metrics like SacreBLEU provide a better metric? What are weaknesses of SacreBLEU? Why don't we use SacreBLEU as our training loss?

(4) Recall that, as our tokenizer doesn't have a `BOS` token, we instead prepend the decoder input with a `PAD` token ("right shift" the decoder input).

In `utils_public/utils_translation_inference.py:predict_translations()`, the code is careful to do the following:
```python
decoder_mask = (target_sequence_tokens != pad_token_id).to(device=device)
# Important: force first token (the PAD acting as a BOS) to be treated as valid.
# Forgetting to do this line significantly harms val/test SacreBLEU scores:
# for instance, for large_v2, test SacreBLEU scores drop from 0.28 to 0.22.
decoder_mask[:, 0] = True
```
Why is this line so important: `decoder_mask[:, 0] = True`?

Hint: consider what happens during training time, and consider that we typically want train and test time conditions to match as closely as possible.

(5) What kind of learning rate schedule did we use in `main_train_multi30k.py`?

(6) In `main_train_multi30k.py`, we use Automatic Mixed Precision ([AMP](https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html)) to accelerate training on GPU cuda hardware.
What is AMP?
Why does it accelerate training?

## (optional) More training!

Finally: you're welcome to train your own transformer models, say with different hyperparameters and such.

However, to avoid cluttering/contaminating your homework submission, I recommend first submitting the homework assignment to Gradescope.
Then, duplicate this directory to a different location in your Google Drive, and train away!

Note that you do have limited GPU compute on Google Colab, and keep in mind there are more homework assignments in this class that will require GPU compute.

# Homework Submission

These cells package the outputs for hw4.ipynb.

To submit your homework assignment, we will do two things:
- (1) Package your code (and any relevant produced artifacts like model weights, `.pth` files, `.pt` files, etc) into a submission zip file
- (2) Download your submission zip file to your local machine
- (3) (your action required) Upload the submission zip file to Gradescope

Once you've submitted to Gradescope, the Gradescope autograder will run and assign your earned points.

Run the below cells to prepare+download your submission zip file:

In [ ]:
# Packages code+artifacts into zip file, and download it to your local machine
# Tip: the zip file will be downloaded to your browser's default download folder, eg `~/Downloads, C:\Users\YourUserName\Downloads`, etc.
# IMPORTANT: be sure that `rootdir_asn` is defined correctly in previous cell
from utils_public.utils import run_cmd

os.chdir(rootdir_asn)
print("cwd: ", os.getcwd())  # make sure we are in the right dir
print("ls cwd: ", os.listdir(os.getcwd()))

zip_outpath = f"{asn_name}_submission.zip"
print(f"Creating zip file (zip_outpath={zip_outpath})...")
run_cmd(["bash", "./utils_public/prepare_submission.sh", zip_outpath])
print("Created zipfile!")

# Check if filesize is too big
filesize_mb = os.path.getsize(zip_outpath) / (1024 * 1024)
print(f"Zip file size: {filesize_mb} MB")
if filesize_mb > 20:
    print(f"Warning: your submission zip is very large, and may result in autograder issues. Please investigate, perhaps you accidentally included unnecessary files?")

# Download created zipfile to your local machine
from google.colab import files
files.download(zip_outpath)
print(
    f"Finished downloading {zip_outpath}! Upload this zip file to Gradescope as your submission to run the autograder. "
    "\nThe zip file will be in your browser's default download directory, eg '~/Downloads', 'C:\\Users\\YourUserName\\Downloads', etc"
)